# Airbnb Price Analytics — Price Model

**Region:** United Kingdom
**Cities:** London and Bristol
**Currency:** GBP

This notebook trains the nightly price model on the Gold feature table produced by the ETL pipeline.

It reserves a fixed test set in each city before any baseline or model is fitted. Preprocessing, the room-type/capacity median baseline and the model use training rows only; the test set is used once, for the final MAE in GBP.

Greater Manchester is not read here. It is scored later by `03_predict` with the frozen model, without retraining.

## 1. Configuration

The catalog, Gold schema, model version and the Volume path for model artifacts are notebook parameters.

Writing to Delta tables is disabled by default during testing.

In [ ]:
import re
import json
from pathlib import Path
import numpy as np
import pandas as pd
import sklearn
from pyspark.sql import functions as F
from sklearn.model_selection import train_test_split
dbutils.widgets.text('catalog', 'workspace')
dbutils.widgets.text('gold_schema', 'airbnb_gold')
dbutils.widgets.text('silver_schema', 'airbnb_silver')
dbutils.widgets.text('model_version', 'v1')
dbutils.widgets.text('model_root', '/Volumes/workspace/airbnb_eda/artifacts/models')
dbutils.widgets.dropdown('write_tables', 'false', ['false', 'true'])
dbutils.widgets.dropdown('run_test_evaluation', 'false', ['false', 'true'])

Widget values are read once and normalised. `run_test_evaluation` stays `false` during development. The test set is evaluated only when both `run_test_evaluation` and `write_tables` are `true`, so its results are always saved.

In [ ]:
catalog = dbutils.widgets.get('catalog').strip()
gold_schema = dbutils.widgets.get('gold_schema').strip()
silver_schema = dbutils.widgets.get('silver_schema').strip()
model_version = dbutils.widgets.get('model_version').strip()
model_root = dbutils.widgets.get('model_root').rstrip('/')
write_tables = dbutils.widgets.get('write_tables') == 'true'
run_test_evaluation = dbutils.widgets.get('run_test_evaluation') == 'true'

Identifiers are validated before they are used in table names. The model version becomes a folder name, and the artifact path must be a Volume in the selected catalog.

In [ ]:
for name, value in [('catalog', catalog), ('gold_schema', gold_schema), ('silver_schema', silver_schema)]:
    if not re.fullmatch('[A-Za-z][A-Za-z0-9_]*', value):
        raise ValueError(f'Invalid {name} identifier')
if not re.fullmatch('[A-Za-z0-9][A-Za-z0-9._-]*', model_version):
    raise ValueError('model_version must be a plain folder name')
if not re.fullmatch(f'/Volumes/{catalog}/[A-Za-z][A-Za-z0-9_]*/[A-Za-z][A-Za-z0-9_]*(/.*)?', model_root):
    raise ValueError('model_root must be a Volume path in the selected catalog')

The training cities, split seed and test share are fixed for this study. Changing them would change the test set, so they are constants rather than widgets.

In [ ]:
TRAINING_CITIES = ['bristol', 'london']
SPLIT_SEED = 42
TEST_SIZE = 0.2
KEY_COLUMNS = ['city', 'listing_id', 'snapshot_date']
features_table = f'{catalog}.{gold_schema}.features'
metrics_table = f'{catalog}.{gold_schema}.model_metrics'
silver_table = f'{catalog}.{silver_schema}.listings'
artifact_dir = Path(model_root) / model_version
print(f'catalog={catalog}, model_version={model_version}, artifact_dir={artifact_dir}, '
      f'write_tables={write_tables}, run_test_evaluation={run_test_evaluation}')

## 2. Load Features

We read the Gold feature table for London and Bristol only. No other city is read in this notebook.

First we check that all expected feature columns exist.

In [ ]:
expected_columns = {
    'city', 'listing_id', 'snapshot_date', 'price_gbp', 'target_valid',
    'room_type', 'property_type', 'accommodates', 'bedrooms', 'beds',
    'bathrooms', 'latitude', 'longitude', 'minimum_nights', 'maximum_nights',
    'feature_version', 'split'
}
features_all = spark.table(features_table)
missing_columns = sorted(expected_columns - set(features_all.columns))
if missing_columns:
    raise ValueError(f'Missing columns in {features_table}: {missing_columns}')
features_df = features_all.filter(F.col('city').isin(TRAINING_CITIES))

Both cities must be present with exactly one snapshot each, and listing keys must be unique. With one snapshot per city, the same listing cannot appear in both the training and test sets.

In [ ]:
snapshot_counts = {r['city']: r['snapshots'] for r in
                   features_df.groupBy('city').agg(F.countDistinct('snapshot_date').alias('snapshots')).collect()}
if sorted(snapshot_counts) != TRAINING_CITIES:
    raise ValueError(f'Expected cities {TRAINING_CITIES}, found {sorted(snapshot_counts)}')
if any(n != 1 for n in snapshot_counts.values()):
    raise ValueError(f'Expected one snapshot per city: {snapshot_counts}')
assert features_df.groupBy(*KEY_COLUMNS).count().filter('count > 1').count() == 0, 'Duplicate listing keys'

The data is small enough for pandas. Rows are sorted by key before splitting: `random_state` only reproduces the same split if the input rows are in the same order, and Spark does not guarantee row order.

In [ ]:
features_pdf = (features_df.drop('split').toPandas()
                .sort_values(KEY_COLUMNS)
                .reset_index(drop=True))
features_pdf['target_valid'] = features_pdf['target_valid'].fillna(False).astype(bool)
print('Rows loaded:', len(features_pdf))
display(features_pdf.groupby('city').agg(listings=('listing_id', 'size'), valid_prices=('target_valid', 'sum')).reset_index())

All rows must share one feature version. It is recorded with the metrics and in the freeze package, and `03_predict` requires the same version.

In [ ]:
feature_versions = list(features_pdf['feature_version'].unique())
assert len(feature_versions) == 1, f'Expected exactly one feature_version, found {feature_versions}'
feature_version = feature_versions[0]
print('feature_version:', feature_version)

## 3. Train/Test Split

Listings with a valid price are split 80/20 into training and test sets, separately in each city and stratified by room type, so both sets have the same room-type mix.

Stratification needs at least two listings per group. A missing room type is treated as its own group, and a room type with a single listing joins the largest group.

In [ ]:
def room_type_strata(room_type):
    labels = room_type.fillna('unknown')
    sizes = labels.map(labels.value_counts())
    return labels.where(sizes >= 2, labels.mode()[0])

Each city is split independently with `random_state=42`. Listings without a valid price are labelled `inference`: they get predictions later but are never used for training or MAE.

In [ ]:
features_pdf['split'] = 'inference'
for city in TRAINING_CITIES:
    valid = features_pdf[(features_pdf['city'] == city) & features_pdf['target_valid']]
    train_idx, test_idx = train_test_split(valid.index, test_size=TEST_SIZE, random_state=SPLIT_SEED,
                                           stratify=room_type_strata(valid['room_type']))
    features_pdf.loc[train_idx, 'split'] = 'train'
    features_pdf.loc[test_idx, 'split'] = 'test'

Every valid-price listing must be in `train` or `test`, every other listing in `inference`, and the test share must be close to 20% in each city.

In [ ]:
valid_mask = features_pdf['target_valid']
assert features_pdf.loc[valid_mask, 'split'].isin(['train', 'test']).all()
assert (features_pdf.loc[~valid_mask, 'split'] == 'inference').all()
test_share = features_pdf[valid_mask].groupby('city')['split'].apply(lambda s: (s == 'test').mean())
assert ((test_share - TEST_SIZE).abs() < 0.01).all(), test_share.to_dict()
print('Test share:', test_share.round(4).to_dict())

The split settings are kept in memory. They are saved later in the JSON manifest next to the model. The scikit-learn version is recorded because a different version could produce a different split from the same seed.

In [ ]:
split_settings = {'split_seed': SPLIT_SEED, 'test_size': TEST_SIZE, 'stratify': 'room_type',
                  'sklearn_version': sklearn.__version__}
print('Split settings:', split_settings)

## 4. Saving the Split

The `split` column of `workspace.airbnb_gold.features` is the single source of truth for split membership. A saved split is never overwritten: if a city already has split values, the recomputed split must match them exactly, otherwise the notebook stops.

First we convert the labels into a Spark DataFrame with an explicit schema.

In [ ]:
split_sdf = spark.createDataFrame(features_pdf[KEY_COLUMNS + ['split']],
                                  schema='city string, listing_id string, snapshot_date date, split string')

This check compares the recomputed split with the saved `features.split` values for a city. Every row of a city with a saved split must match, so a remaining empty value also counts as a difference. It only reads the table, so it also runs in dry-run mode.

In [ ]:
def saved_split_status(city):
    saved = spark.table(features_table).filter(F.col('city') == city).select(*KEY_COLUMNS, 'split')
    if saved.filter(F.col('split').isNotNull()).limit(1).count() == 0:
        return 'new'
    current = split_sdf.filter(F.col('city') == city)
    if saved.exceptAll(current).limit(1).count() or current.exceptAll(saved).limit(1).count():
        raise ValueError(f'Saved split for {city} differs from the recomputed split; a published split must not change')
    return 'verified'

Each city is either `new` (no saved split yet) or `verified` (the saved split matches).

In [ ]:
split_status = {city: saved_split_status(city) for city in TRAINING_CITIES}
print('Split status:', split_status)

Labels are written into the existing `split` column with `MERGE`, only for new cities and only where the value is empty. The table schema is not changed, so the ETL pipeline can still verify the published features.

In [ ]:
def fill_feature_split(cities):
    split_sdf.filter(F.col('city').isin(cities)).createOrReplaceTempView('price_model_split')
    spark.sql(f'''
        MERGE INTO {features_table} AS t USING price_model_split AS s
        ON t.city = s.city AND t.listing_id = s.listing_id AND t.snapshot_date = s.snapshot_date
        WHEN MATCHED AND t.split IS NULL THEN UPDATE SET t.split = s.split''')

The feature table is updated only when writing is enabled. Verified cities are left unchanged.

In [ ]:
new_cities = [c for c, status in split_status.items() if status == 'new']
if write_tables and new_cities:
    fill_feature_split(new_cities)
    print('Split written to', features_table, 'for', new_cities)
elif write_tables:
    print('All saved splits verified; nothing written')
else:
    print('DRY RUN: nothing written. Set widget write_tables=true only after review.')

After writing, the `split` column is read back and compared with the computed split.

In [ ]:
if write_tables:
    stored = spark.table(features_table).filter(F.col('city').isin(TRAINING_CITIES)).select(*KEY_COLUMNS, 'split')
    assert stored.exceptAll(split_sdf).limit(1).count() == 0
    assert split_sdf.exceptAll(stored).limit(1).count() == 0
    print(f'Validated persisted split for {TRAINING_CITIES}')
else:
    print('DRY RUN validated in memory')

## 5. Split Summary

Listing counts per city and split. The test share is calculated over listings with a valid price.

In [ ]:
split_summary = (pd.crosstab(features_pdf['city'], features_pdf['split'])
                 .reindex(columns=['train', 'test', 'inference'], fill_value=0))
split_summary['test_share'] = (split_summary['test'] / (split_summary['train'] + split_summary['test'])).round(4)
display(split_summary.reset_index())

The following sections fit everything on `train_pdf` only. The test rows stay in `features_pdf` and are not used until the final evaluation.

In [ ]:
train_pdf = features_pdf[features_pdf['split'] == 'train'].reset_index(drop=True)
print('Training rows:', len(train_pdf))

## 6. Baseline

The required baseline is the training median price by room type and guest capacity (`accommodates`).

The statistics are computed on training rows only and pooled across London and Bristol, so the same table can be applied unchanged to a new city. The fallbacks for unseen combinations are fixed now: the room-type training median, then the global training median.

`fit_baseline` computes the three levels. A missing room type or capacity does not form its own group; such listings use the next fallback level. The statistics are plain Python values, so they can be saved as JSON with the model.

In [ ]:
BASELINE_LEVELS = ['group', 'room_type', 'global']

def fit_baseline(train):
    rows = train[['room_type', 'accommodates', 'price_gbp']].astype({'accommodates': 'Int64'})
    group = rows.groupby(['room_type', 'accommodates'])['price_gbp'].agg(['median', 'size']).reset_index()
    room = rows.groupby('room_type')['price_gbp'].agg(['median', 'size']).reset_index()
    return {
        'levels': BASELINE_LEVELS,
        'group_medians': [{'room_type': r, 'accommodates': int(a), 'median_price_gbp': float(m), 'listings': int(n)}
                          for r, a, m, n in group.itertuples(index=False)],
        'room_type_medians': [{'room_type': r, 'median_price_gbp': float(m), 'listings': int(n)}
                              for r, m, n in room.itertuples(index=False)],
        'global_median_price_gbp': float(rows['price_gbp'].median()),
        'train_listings': int(len(rows)),
    }

The baseline is fitted on `train_pdf`. The check confirms that it contains only training rows with a valid price, so no test or inference listing influences the statistics.

In [ ]:
assert (train_pdf['split'] == 'train').all() and train_pdf['target_valid'].all()
assert train_pdf['price_gbp'].notna().all()
baseline_stats = fit_baseline(train_pdf)
print(f"Groups: {len(baseline_stats['group_medians'])}, room types: {len(baseline_stats['room_type_medians'])}, "
      f"global median: £{baseline_stats['global_median_price_gbp']:.2f}")

`predict_baseline` applies saved statistics to any listings, including a new city. It returns a copy with `baseline_price_gbp` and `baseline_level` (`group`, `room_type` or `global`), keeping the input row order. `03_predict` reuses this function with the saved statistics.

In [ ]:
def predict_baseline(df, stats):
    keys = df[['room_type', 'accommodates']].astype({'accommodates': 'Int64'})
    group = pd.DataFrame(stats['group_medians'], columns=['room_type', 'accommodates', 'median_price_gbp', 'listings'])
    room = pd.DataFrame(stats['room_type_medians'], columns=['room_type', 'median_price_gbp', 'listings'])
    group_price = keys.merge(group.astype({'accommodates': 'Int64'}), how='left', on=['room_type', 'accommodates'],
                             validate='many_to_one')['median_price_gbp'].to_numpy(dtype=float)
    room_price = keys[['room_type']].merge(room, how='left', on='room_type',
                                           validate='many_to_one')['median_price_gbp'].to_numpy(dtype=float)
    has_group, has_room = ~np.isnan(group_price), ~np.isnan(room_price)
    out = df.copy()
    out['baseline_price_gbp'] = np.select([has_group, has_room], [group_price, room_price], stats['global_median_price_gbp'])
    out['baseline_level'] = np.select([has_group, has_room], ['group', 'room_type'], 'global')
    return out

The statistics stay in memory for now and are saved with the model later. This check confirms that they serialise to JSON.

In [ ]:
baseline_json = json.dumps(baseline_stats)
print('Baseline statistics serialise to JSON:', len(baseline_json), 'characters')

As a sanity check, the baseline is applied to the training rows. Train MAE is not a performance estimate: the test set is evaluated only at the end.

In [ ]:
train_baseline = predict_baseline(train_pdf, baseline_stats)
assert len(train_baseline) == len(train_pdf) and train_baseline['baseline_price_gbp'].notna().all()
train_baseline['abs_error'] = (train_baseline['price_gbp'] - train_baseline['baseline_price_gbp']).abs()

MAE in GBP per city and combined. The combined value is calculated over all evaluated listings, so it is weighted by listing count rather than averaging the city MAEs. The same function is reused for the final evaluation.

In [ ]:
def mae_table(frame):
    by_city = frame.groupby('city').agg(listings=('abs_error', 'size'), mae_gbp=('abs_error', 'mean'))
    combined = pd.DataFrame({'listings': [len(frame)], 'mae_gbp': [frame['abs_error'].mean()]}, index=['combined'])
    return pd.concat([by_city, combined]).round({'mae_gbp': 2}).rename_axis('city').reset_index()

display(mae_table(train_baseline))

Fallback usage on the training rows. Most listings should use the group median; the other levels apply only where room type or capacity is missing.

In [ ]:
display(pd.crosstab(train_baseline['city'], train_baseline['baseline_level'])
        .reindex(columns=BASELINE_LEVELS, fill_value=0).reset_index())

## 7. Model Selection

Two candidate models are compared with the baseline using 5-fold cross-validation on the training rows only. The test set is not used.

- **Target:** the models learn `log1p(price_gbp)`, which reduces the influence of the long upper price tail. Predictions are converted back to GBP with `expm1`, and every MAE is reported in GBP.
- **Features:** an explicit allowlist from the cleaning specification. `city` is not a feature, so the model can be applied to a new city.
- **Candidates:** Ridge regression and a histogram gradient boosting model (HGB). Each is an sklearn Pipeline, so category grouping, encoding and imputation are refitted inside every fold.

Model imports, the feature allowlist and the fixed cross-validation folds. All candidates use the same folds, so their scores are directly comparable.

In [ ]:
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.model_selection import KFold, ParameterGrid
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler

`MIN_CATEGORY_COUNT` is fixed before cross-validation and is not tuned.

In [ ]:
CATEGORICAL = ['room_type', 'property_type']
NUMERIC = ['accommodates', 'bedrooms', 'beds', 'bathrooms', 'latitude', 'longitude', 'minimum_nights', 'maximum_nights']
FEATURES = CATEGORICAL + NUMERIC   # explicit allowlist; city is deliberately excluded
MIN_CATEGORY_COUNT = 20            # categories with fewer training rows are grouped as 'Other'
assert (train_pdf['split'] == 'train').all() and set(FEATURES) <= set(train_pdf.columns)
cv_folds = list(KFold(n_splits=5, shuffle=True, random_state=SPLIT_SEED).split(train_pdf))

Property types with fewer than `MIN_CATEGORY_COUNT` training rows are grouped as `Other`. Values unseen during fitting and missing values are also mapped to `Other`, so a new city cannot produce an unknown category.

The same rule is applied to `room_type`. All four room types are common in training, so the rule does not change them here, but it gives an unseen or missing room type in a new city the same `Other` handling.

In [ ]:
class RareCategoryGrouper(BaseEstimator, TransformerMixin):
    """Replace rare, unseen and missing category values with 'Other'."""

    def __init__(self, columns, min_count):
        self.columns = columns
        self.min_count = min_count

    def fit(self, X, y=None):
        # Allowed categories are learned from the fitting rows only.
        self.allowed_ = {c: sorted(X[c].value_counts().loc[lambda s: s >= self.min_count].index) for c in self.columns}
        return self

    def transform(self, X):
        X = X.copy()
        for c in self.columns:
            X[c] = X[c].where(X[c].isin(self.allowed_[c]), 'Other').astype(object)
        return X

**Candidate 1: Ridge.** Categories are one-hot encoded. Numeric features use median imputation with missing-value indicators, then scaling so that the Ridge penalty treats features comparably.

In [ ]:
def make_ridge():
    preprocess = ColumnTransformer([
        ('cat', OneHotEncoder(handle_unknown='ignore'), CATEGORICAL),
        ('num', make_pipeline(SimpleImputer(strategy='median', add_indicator=True), StandardScaler()), NUMERIC),
    ])
    pipeline = Pipeline([('group', RareCategoryGrouper(CATEGORICAL, MIN_CATEGORY_COUNT)),
                         ('prep', preprocess), ('model', Ridge(alpha=1.0))])
    # Fits on log1p(price); predict() returns GBP through expm1.
    return TransformedTargetRegressor(regressor=pipeline, func=np.log1p, inverse_func=np.expm1)

**Candidate 2: HGB.** Categories become integer codes for the native categorical support. Missing numeric values are left as NaN, which HGB handles directly. A small grid of learning rates and tree sizes is tuned.

Early stopping holds out a random 10% of the fitting rows, never the validation fold, so `random_state` is fixed for reproducibility.

In [ ]:
HGB_GRID = ParameterGrid({'learning_rate': [0.05, 0.1], 'max_leaf_nodes': [31, 63]})

def make_hgb(learning_rate, max_leaf_nodes):
    preprocess = ColumnTransformer([
        # A category unseen during fitting would become NaN; after grouping this can only be 'Other'.
        ('cat', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=np.nan), CATEGORICAL),
        ('num', 'passthrough', NUMERIC),
    ])
    model = HistGradientBoostingRegressor(
        categorical_features=[0, 1],  # positions of CATEGORICAL in the ColumnTransformer output
        learning_rate=learning_rate, max_leaf_nodes=max_leaf_nodes, max_iter=500,
        early_stopping=True, random_state=SPLIT_SEED)
    pipeline = Pipeline([('group', RareCategoryGrouper(CATEGORICAL, MIN_CATEGORY_COUNT)),
                         ('prep', preprocess), ('model', model)])
    return TransformedTargetRegressor(regressor=pipeline, func=np.log1p, inverse_func=np.expm1)

Cross-validation produces out-of-fold predictions: each training listing is predicted by a model fitted without it. The baseline is refitted inside each fold in the same way, so all three candidates are compared on equal terms.

In [ ]:
def cv_predictions(fit_predict):
    oof = np.full(len(train_pdf), np.nan)
    for fit_idx, val_idx in cv_folds:
        oof[val_idx] = fit_predict(train_pdf.iloc[fit_idx], train_pdf.iloc[val_idx])
    assert not np.isnan(oof).any(), 'Every training row must receive one out-of-fold prediction'
    return oof

def baseline_fit_predict(fit_rows, val_rows):
    return predict_baseline(val_rows, fit_baseline(fit_rows))['baseline_price_gbp'].to_numpy()

For a model, each fold fits a new pipeline on the fitting rows and predicts the held-out rows in GBP.

In [ ]:
def model_fit_predict(make_model):
    def fit_predict(fit_rows, val_rows):
        model = make_model().fit(fit_rows[FEATURES], fit_rows['price_gbp'])
        return model.predict(val_rows[FEATURES])
    return fit_predict

Out-of-fold errors are summarised with `mae_table`: MAE in GBP for London, Bristol and all listings pooled.

In [ ]:
def cv_mae(candidate, oof):
    errors = pd.DataFrame({'city': train_pdf['city'], 'abs_error': (train_pdf['price_gbp'] - oof).abs()})
    return mae_table(errors).assign(candidate=candidate)

Cross-validation for the baseline and Ridge.

In [ ]:
cv_results = [cv_mae('baseline', cv_predictions(baseline_fit_predict)),
              cv_mae('ridge', cv_predictions(model_fit_predict(make_ridge)))]

Cross-validation for each HGB setting in the grid, ordered by pooled MAE.

In [ ]:
hgb_results, hgb_params = [], {}
for params in HGB_GRID:
    name = f"hgb lr={params['learning_rate']} leaves={params['max_leaf_nodes']}"
    hgb_params[name] = params
    hgb_results.append(cv_mae(name, cv_predictions(model_fit_predict(lambda: make_hgb(**params)))))
hgb_cv = pd.concat(hgb_results).pivot(index='candidate', columns='city', values='mae_gbp')
display(hgb_cv[TRAINING_CITIES + ['combined']].sort_values('combined').reset_index())

The comparison table shows the baseline, Ridge and the best HGB setting. Every candidate is evaluated on the same listings; the counts are shown below the table.

The baseline is the comparator, not a candidate: the final model is Ridge or the best HGB setting, whichever has the lower pooled cross-validated MAE. It is refitted on all training rows in the next section.

In [ ]:
best_hgb = hgb_cv['combined'].idxmin()
cv_table = pd.concat(cv_results + [r for r in hgb_results if r['candidate'].iat[0] == best_hgb])
cv_summary = cv_table.pivot(index='candidate', columns='city', values='mae_gbp')[TRAINING_CITIES + ['combined']]
display(cv_summary.sort_values('combined').reset_index())
display(cv_results[0][['city', 'listings']])

The selected model and its settings are kept in memory for the final fit. If it does not beat the baseline in cross-validation, a warning is printed and the notebook continues.

In [ ]:
selected_candidate = cv_summary.drop(index='baseline')['combined'].idxmin()
selected_params = hgb_params.get(selected_candidate, {})
print('Selected by pooled CV MAE:', selected_candidate, selected_params)
selected_cv_mae, baseline_cv_mae = cv_summary.loc[selected_candidate, 'combined'], cv_summary.loc['baseline', 'combined']
if selected_cv_mae >= baseline_cv_mae:
    print(f'WARNING: {selected_candidate} does not beat the baseline in CV '
          f'(pooled MAE £{selected_cv_mae:.2f} vs baseline £{baseline_cv_mae:.2f}).')

## 8. Final Model

The selected model is refitted on all training rows with its selected settings. This step does not use the test set, so it also runs when test evaluation is disabled.

In [ ]:
model_family = 'ridge' if selected_candidate == 'ridge' else 'hgb'
final_model = make_ridge() if model_family == 'ridge' else make_hgb(**selected_params)
final_model.fit(train_pdf[FEATURES], train_pdf['price_gbp'])
print(f'Fitted {selected_candidate} on {len(train_pdf)} training rows')

The category lists learned by the final model are saved in the freeze package. Any other value, in A/B or a new city, is mapped to `Other`.

In [ ]:
category_lists = final_model.regressor_.named_steps['group'].allowed_
for column, values in category_lists.items():
    print(f'{column}: {len(values)} categories kept')

## 9. Pre-flight Checks

Everything that can be checked without test rows is checked here, before the test gate, so that a failure after the test set has been read is as unlikely as possible. These cells read Silver and, when the test evaluation will run, prepare the model Volume. They do not read test rows.

First, the imports and constants used by the remaining sections.

In [ ]:
import io, sys, hashlib, uuid
from datetime import datetime, timezone
import joblib
import matplotlib.pyplot as plt
from sklearn.inspection import permutation_importance
DATASET = 'development'
PACKAGE_FILES = ['model.joblib', 'baseline_stats.json', 'check_set.json']
manifest_path = artifact_dir / 'manifest.json'

`spark_ready` turns NaN into `None`, so missing values are stored as NULL rather than NaN. `to_records` converts DataFrames to plain JSON records with full float precision. `sha256_file` checksums package files.

In [ ]:
def spark_ready(frame):
    return frame.astype(object).where(frame.notna(), None)

def to_records(frame):
    return json.loads(frame.to_json(orient='records', double_precision=15))

def sha256_file(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()

The `model_metrics` layout: one row per predictor and metric, with the evaluated count (MAE denominator) and the eligible count (coverage denominator). `rows_loaded` and `source_sha256` describe holdout runs from `03_predict` and are empty here.

In [ ]:
metrics_columns = ['model_version', 'model_name', 'feature_version', 'dataset', 'split', 'city', 'predictor', 'metric',
                   'value', 'evaluated_listings', 'eligible_listings', 'rows_loaded', 'source_sha256']
metrics_schema = ('model_version string, model_name string, feature_version string, dataset string, split string, '
                  'city string, predictor string, metric string, value double, evaluated_listings long, '
                  'eligible_listings long, rows_loaded long, source_sha256 string')

def metric_rows(table, split):
    metrics = [m for m in ['mae_gbp', 'median_ae_gbp', 'prediction_coverage'] if m in table.columns]
    rows = table.melt(id_vars=['predictor', 'city', 'listings', 'eligible_listings'], value_vars=metrics, var_name='metric')
    return (rows.rename(columns={'listings': 'evaluated_listings'})
            .assign(split=split, model_version=model_version, model_name=selected_candidate, feature_version=feature_version,
                    dataset=DATASET, rows_loaded=None, source_sha256=None))

Cross-validation rows keep only the baseline and the selected model, labelled as `baseline` and `model`. Every training row receives an out-of-fold prediction, so the eligible and evaluated counts are equal. Converting them to Spark now checks the schema and value types before any test row is read.

In [ ]:
cv_rows = cv_table[cv_table['candidate'].isin(['baseline', selected_candidate])].copy()
cv_rows['predictor'] = np.where(cv_rows['candidate'] == 'baseline', 'baseline', 'model')
cv_rows['eligible_listings'] = cv_rows['listings']
cv_metrics_pdf = metric_rows(cv_rows, 'cv')
assert spark.createDataFrame(spark_ready(cv_metrics_pdf[metrics_columns]), schema=metrics_schema).count() == len(cv_metrics_pdf)

If `model_metrics` already exists, its columns must match this layout, or the later write would fail.

In [ ]:
if spark.catalog.tableExists(metrics_table):
    existing_columns = set(spark.table(metrics_table).columns)
    expected_columns = set(metrics_columns) | {'generated_at'}
    if existing_columns != expected_columns:
        raise ValueError(f'{metrics_table} columns differ: missing {sorted(expected_columns - existing_columns)}, '
                         f'unexpected {sorted(existing_columns - expected_columns)}')

Source provenance comes from Silver: one source file, checksum and pipeline version per training city.

In [ ]:
sources = [r.asDict() for r in (spark.table(silver_table).filter(F.col('city').isin(TRAINING_CITIES))
           .select('city', F.col('snapshot_date').cast('string').alias('snapshot_date'),
                   'source_url', 'source_sha256', 'pipeline_version')
           .distinct().orderBy('city').collect())]
assert len(sources) == len(TRAINING_CITIES), 'Expected one Silver source and pipeline version per city'

The check set lets `03_predict` detect drift in its copies of `predict_baseline` and `RareCategoryGrouper`, or in the environment. It holds 14 sampled training rows plus 6 modified copies that force the fallback levels and an unseen property type. It stores features only: no keys or prices.

In [ ]:
check_rows = train_pdf.sample(14, random_state=SPLIT_SEED)[FEATURES].reset_index(drop=True)
variants = check_rows.head(6).astype({'accommodates': 'float64'})
variants.loc[0:1, 'accommodates'] = np.nan                 # missing capacity -> room_type level
variants.loc[2, 'accommodates'] = 99                       # unseen capacity -> room_type level
variants.loc[3, 'room_type'] = None                        # missing room type -> global level
variants.loc[4, 'room_type'] = 'Unseen room type'          # unseen room type -> global level
variants.loc[5, 'property_type'] = 'Unseen property type'  # grouped as Other by the model
check_set = pd.concat([check_rows.astype({'accommodates': 'float64'}), variants], ignore_index=True)

The expected outputs are computed with the in-memory baseline and model. The check set must cover all three baseline levels.

In [ ]:
check_expected = predict_baseline(check_set, baseline_stats)
check_expected['predicted_price_gbp'] = final_model.predict(check_set[FEATURES])
assert set(check_expected['baseline_level']) == set(BASELINE_LEVELS), 'Check set must cover every baseline level'
check_set_doc = {'features': FEATURES, 'tolerance': 1e-6,
                 'rows': to_records(check_expected[FEATURES + ['baseline_level', 'baseline_price_gbp', 'predicted_price_gbp']])}
display(check_expected[FEATURES + ['baseline_level', 'baseline_price_gbp', 'predicted_price_gbp']])

The package is rehearsed in memory: the model goes through joblib and the statistics and check set through JSON, then the check set must reproduce the expected predictions. This is the same verification that runs on the saved files after the gate.

In [ ]:
buffer = io.BytesIO()
joblib.dump(final_model, buffer)
buffer.seek(0)
rehearsed_model = joblib.load(buffer)
rehearsed_stats = json.loads(json.dumps(baseline_stats))
rehearsed_rows = pd.DataFrame(json.loads(json.dumps(check_set_doc))['rows'])
assert np.allclose(predict_baseline(rehearsed_rows, rehearsed_stats)['baseline_price_gbp'], rehearsed_rows['baseline_price_gbp'], rtol=1e-6)
assert np.allclose(rehearsed_model.predict(rehearsed_rows[FEATURES]), rehearsed_rows['predicted_price_gbp'], rtol=1e-6)
print('Package rehearsal passed')

The manifest records how the model was built and evaluated. Everything except the test metrics is known now, so it is built and checked for JSON serialisation here; the test metrics, file checksums and creation time are added when the package is saved.

In [ ]:
manifest = {
    'model_version': model_version, 'model_name': selected_candidate, 'model_family': model_family,
    'model_params': selected_params, 'target': 'log1p(price_gbp); predictions are returned in GBP via expm1',
    'feature_version': feature_version, 'features': {'categorical': CATEGORICAL, 'numeric': NUMERIC},
    'min_category_count': MIN_CATEGORY_COUNT, 'category_lists': category_lists,
    'training_cities': TRAINING_CITIES, 'sources': sources,
    'split': {**split_settings, 'counts': to_records(split_summary.reset_index())},
    'cv_mae_gbp': to_records(cv_summary.reset_index()),
    'environment': {'python': sys.version.split()[0], 'sklearn': sklearn.__version__, 'pandas': pd.__version__,
                    'numpy': np.__version__, 'joblib': joblib.__version__},
}
json.dumps(manifest)

When the test evaluation will run, the model Volume is prepared now: the version must not be frozen yet, and a probe file confirms that the package folder is writable. An existing Volume is used as it is; `CREATE VOLUME` runs only when the Volume in `model_root` is missing, because it needs a privilege that is not always granted.

In [ ]:
def prepare_model_volume():
    parts = model_root.split('/')  # ['', 'Volumes', catalog, schema, volume, ...]
    volume_name, volume_root = '.'.join(parts[2:5]), Path('/'.join(parts[:5]))
    if not volume_root.is_dir():
        try:
            spark.sql(f'CREATE VOLUME IF NOT EXISTS {volume_name}')
        except Exception as error:
            raise PermissionError(f'Volume {volume_name} does not exist and cannot be created: {error}') from error
    try:
        artifact_dir.mkdir(parents=True, exist_ok=True)
        probe = artifact_dir / '.write_check'
        probe.write_text('ok', encoding='utf-8')
        assert probe.read_text(encoding='utf-8') == 'ok'
        probe.unlink()
    except OSError as error:
        raise PermissionError(f'Cannot write the freeze package to {artifact_dir}: {error}') from error

The Volume is prepared only when the test evaluation will run.

In [ ]:
if run_test_evaluation and write_tables:
    if manifest_path.exists():
        raise ValueError(f'{model_version} is already frozen ({manifest_path}); use a new model_version')
    prepare_model_volume()
    print('Model Volume writable:', artifact_dir)
else:
    print('Volume check skipped: the notebook stops at the test gate')

When the test evaluation will run, a tiny temporary Delta table with a unique name is also created in the Gold schema and dropped again. This confirms that `model_metrics` can be created there before any test row is read.

In [ ]:
def check_table_privilege():
    probe_table = f'{catalog}.{gold_schema}.tmp_write_check_{uuid.uuid4().hex[:12]}'
    try:
        spark.createDataFrame([(1,)], 'probe int').write.format('delta').saveAsTable(probe_table)
    except Exception as error:
        raise PermissionError(f'Cannot create tables in {catalog}.{gold_schema}; USE SCHEMA and CREATE TABLE '
                              f'privileges are required. Spark error: {error}') from error
    spark.sql(f'DROP TABLE IF EXISTS {probe_table}')

if run_test_evaluation and write_tables:
    check_table_privilege()
    print(f'Table creation in {catalog}.{gold_schema} verified')
else:
    print('Table privilege check skipped: the notebook stops at the test gate')

## 10. Test Evaluation

The test set is used once, after the model and baseline are fixed. Nothing in this or the following sections changes the model.

The remaining sections run only when both `run_test_evaluation=true` and `write_tables=true`; otherwise the notebook stops here, before any test row is read. The test set is looked at once, so its metrics and the freeze package are always saved.

A model version that already has a saved freeze package is not evaluated again: a new evaluation needs a new `model_version`.

In [ ]:
if not (run_test_evaluation and write_tables):
    print('Stopped before test evaluation. Set both run_test_evaluation=true and write_tables=true: '
          'the test set is evaluated once and its results are always saved.')
    dbutils.notebook.exit('Stopped before test evaluation')
if manifest_path.exists():
    raise ValueError(f'{model_version} is already frozen ({manifest_path}); use a new model_version')

The baseline and the final model predict every test listing.

In [ ]:
assert run_test_evaluation and write_tables, 'Test rows are used only when run_test_evaluation and write_tables are true'
test_pdf = features_pdf[features_pdf['split'] == 'test'].reset_index(drop=True)
test_eval = predict_baseline(test_pdf, baseline_stats)
test_eval['predicted_price_gbp'] = final_model.predict(test_eval[FEATURES])
print('Test listings:', len(test_eval))

### 10.1 Test Metrics

For each city and pooled: MAE and median absolute error in GBP, and prediction coverage. A prediction counts as covered when it is a finite, non-negative price. Errors are computed only for covered listings: `listings` is that evaluated count, and `eligible_listings` (all test listings with a valid target) is the coverage denominator.

In [ ]:
def city_rows(frame, city):
    return frame if city == 'combined' else frame[frame['city'] == city]

def test_summary(frame, prediction_col, predictor):
    frame = frame.assign(covered=np.isfinite(frame[prediction_col]) & (frame[prediction_col] >= 0))
    errors = frame[frame['covered']].assign(abs_error=(frame['price_gbp'] - frame[prediction_col]).abs())
    table = mae_table(errors)
    table['median_ae_gbp'] = [round(city_rows(errors, c)['abs_error'].median(), 2) for c in table['city']]
    table['prediction_coverage'] = [round(city_rows(frame, c)['covered'].mean(), 4) for c in table['city']]
    table['eligible_listings'] = [len(city_rows(frame, c)) for c in table['city']]
    return table.assign(predictor=predictor)

Test metrics for the baseline and the model.

In [ ]:
test_results = pd.concat([test_summary(test_eval, 'baseline_price_gbp', 'baseline'),
                          test_summary(test_eval, 'predicted_price_gbp', 'model')], ignore_index=True)
display(test_results[['predictor', 'city', 'listings', 'eligible_listings', 'mae_gbp', 'median_ae_gbp', 'prediction_coverage']])

The headline comparison: how much lower the model's test MAE is than the baseline's.

In [ ]:
test_mae = test_results.pivot(index='city', columns='predictor', values='mae_gbp')
for city in TRAINING_CITIES + ['combined']:
    model_mae, base_mae = test_mae.loc[city, 'model'], test_mae.loc[city, 'baseline']
    change = model_mae / base_mae - 1
    print(f"{city}: model MAE £{model_mae:.2f} vs baseline £{base_mae:.2f} "
          f"({abs(change):.1%} {'lower' if change <= 0 else 'higher'})")

### 10.2 Error by Room Type

Test MAE per room type, pooled across both cities, for the baseline and the model. Small room-type groups give unstable estimates, so the listing count is shown.

In [ ]:
test_errors = test_eval.assign(baseline_abs_error=(test_eval['price_gbp'] - test_eval['baseline_price_gbp']).abs(),
                               model_abs_error=(test_eval['price_gbp'] - test_eval['predicted_price_gbp']).abs())
by_room_type = (test_errors.groupby('room_type', dropna=False)
                .agg(listings=('listing_id', 'size'), baseline_mae_gbp=('baseline_abs_error', 'mean'),
                     model_mae_gbp=('model_abs_error', 'mean'))
                .round(2).sort_values('listings', ascending=False))
display(by_room_type.reset_index())

### 10.3 Permutation Importance

Each feature is shuffled in the test rows five times and the model's MAE is recomputed. The increase in MAE (GBP) shows how much the model relies on that feature. This describes the fixed model and is not used to change it.

In [ ]:
importance = permutation_importance(final_model, test_eval[FEATURES], test_eval['price_gbp'],
                                    scoring='neg_mean_absolute_error', n_repeats=5, random_state=SPLIT_SEED)
# With a negated-MAE score, each importance is the increase in MAE (GBP) when the feature is shuffled.
importance_df = (pd.DataFrame({'feature': FEATURES, 'mae_increase_gbp': importance.importances_mean,
                               'std_gbp': importance.importances_std})
                 .sort_values('mae_increase_gbp'))

The chart orders features by importance; the error bars show the spread across the five shuffles.

In [ ]:
top = importance_df.iloc[-1]
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.barh(importance_df['feature'], importance_df['mae_increase_gbp'], height=0.6, color='#2a78d6',
        xerr=importance_df['std_gbp'], error_kw={'ecolor': '#52514e', 'elinewidth': 1})
ax.set_title(f"{top['feature']} matters most: shuffling it adds £{top['mae_increase_gbp']:.0f} to test MAE",
             loc='left', fontsize=12, color='#0b0b0b')
ax.set_xlabel('Increase in test MAE when the feature is shuffled (GBP)', color='#52514e')
ax.spines[['top', 'right']].set_visible(False)
ax.spines[['left', 'bottom']].set_color('#c3c2b7')
ax.tick_params(colors='#52514e')
ax.grid(axis='x', color='#e6e5e1', linewidth=0.8)
ax.set_axisbelow(True)
plt.tight_layout()
plt.show()

### 10.4 Largest Test Errors

The 20 test listings with the largest model errors, with their attributes. Extreme prices are retained under the cleaning specification, so some large errors may come from prices that are unusual rather than wrong. Listing IDs appear only in notebook output, which is cleared before commit.

In [ ]:
top_errors = test_errors.nlargest(20, 'model_abs_error')[[
    'city', 'listing_id', 'room_type', 'property_type', 'accommodates', 'bedrooms', 'bathrooms',
    'price_gbp', 'predicted_price_gbp', 'baseline_price_gbp', 'model_abs_error']]
display(top_errors.round(2))

## 11. Model Metrics Table

Metrics are written to `workspace.airbnb_gold.model_metrics` in the layout defined in section 9: the cross-validated MAE of the baseline and the selected model, and all test metrics.

`dataset = 'development'` marks London/Bristol results. A rerun replaces only the development rows of this model version, so third-city results written later for the same version are never touched.

In [ ]:
metrics_pdf = pd.concat([cv_metrics_pdf, metric_rows(test_results, 'test')], ignore_index=True)
metrics_sdf = (spark.createDataFrame(spark_ready(metrics_pdf[metrics_columns]), schema=metrics_schema)
               .withColumn('generated_at', F.current_timestamp()))
display(metrics_sdf)

The table is written, replacing this model version's development rows, and read back.

In [ ]:
metrics_predicate = f"model_version = '{model_version}' AND dataset = '{DATASET}'"
if not spark.catalog.tableExists(metrics_table):
    metrics_sdf.write.format('delta').saveAsTable(metrics_table)
else:
    metrics_sdf.write.format('delta').mode('overwrite').option('replaceWhere', metrics_predicate).saveAsTable(metrics_table)
assert spark.table(metrics_table).filter(metrics_predicate).count() == len(metrics_pdf)
print('Written:', metrics_table)

## 12. Freeze Package

The freeze package in `model_root/<model_version>/` holds everything `03_predict` needs to score a new city without refitting:

- `model.joblib`: the fitted model pipeline;
- `baseline_stats.json`: the baseline medians and fallbacks;
- `check_set.json`: a small set of feature rows with their expected baseline and model predictions;
- `manifest.json`: feature allowlist, category lists, split settings, sources, metrics, environment and file checksums.

The Volume was prepared in section 9. Files of an unfinished package without a manifest may be replaced; the manifest is written last, with exclusive mode, and never overwritten: its presence marks the model version as frozen.

In [ ]:
def write_package_files():
    artifact_dir.mkdir(parents=True, exist_ok=True)
    joblib.dump(final_model, artifact_dir / 'model.joblib')
    (artifact_dir / 'baseline_stats.json').write_text(json.dumps(baseline_stats, indent=2), encoding='utf-8')
    (artifact_dir / 'check_set.json').write_text(json.dumps(check_set_doc, indent=2), encoding='utf-8')

The manifest gains the test metrics and the checksum of every package file.

In [ ]:
def write_manifest():
    files = {name: sha256_file(artifact_dir / name) for name in PACKAGE_FILES}
    final_manifest = {**manifest, 'test_metrics': to_records(test_results), 'files': files,
                      'created_at': datetime.now(timezone.utc).isoformat()}
    with manifest_path.open('x', encoding='utf-8') as f:
        json.dump(final_manifest, f, indent=2)

write_package_files()
write_manifest()
print('Freeze package saved:', artifact_dir)

The saved package is loaded back the way `03_predict` will load it. The checksums must match, and the check set must reproduce the expected baseline and model predictions.

In [ ]:
saved_manifest = json.loads(manifest_path.read_text(encoding='utf-8'))
assert all(sha256_file(artifact_dir / n) == h for n, h in saved_manifest['files'].items()), 'Checksum mismatch'
loaded_model = joblib.load(artifact_dir / 'model.joblib')
loaded_stats = json.loads((artifact_dir / 'baseline_stats.json').read_text(encoding='utf-8'))
checks = json.loads((artifact_dir / 'check_set.json').read_text(encoding='utf-8'))
rows = pd.DataFrame(checks['rows'])
assert np.allclose(predict_baseline(rows, loaded_stats)['baseline_price_gbp'], rows['baseline_price_gbp'], rtol=checks['tolerance'])
assert np.allclose(loaded_model.predict(rows[FEATURES]), rows['predicted_price_gbp'], rtol=checks['tolerance'])
print(f'Freeze package verified: {model_version}')